High-level flow implemented below:

load CSV(s)

inspect and basic cleaning

feature selection (simple; drop obvious non-numeric IDs)

train/test split with stratification

baseline ML: RandomForest (report accuracy, F1, ROC AUC)

simple Keras NN (dense) — train and plot loss/accuracy & ROC/PR

visualize class balance, correlation heatmap, feature distributions, confusion matrix

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from helper_functions import *

In [ ]:
# SETTINGS
DATA_DIR = "data"
CSV_GLOB = os.path.join(DATA_DIR, "*.csv")  # adjust if you have one combined CSV
LABEL_COL = 'Label'   # common name in many CIC-DDoS2019 CSVs; adjust if different
RANDOM_STATE = 42

In [ ]:
# LOAD DATA 
csv_paths = sorted(glob.glob(CSV_GLOB))
if not csv_paths:
    raise FileNotFoundError(f"No CSVs found in {DATA_DIR}. Download CIC-DDoS2019 and place CSV(s) there.")
print("Found CSVs:", csv_paths[:5])
df = load_csvs_glob(csv_paths) if len(csv_paths) > 1 else pd.read_csv(csv_paths[0])
print("Initial shape:", df.shape)

inspect_df(df)

In [ ]:
# BASIC CLEANING
drop_cols = ['Flow ID', 'Timestamp', 'Source IP', 'Destination IP']  # drop IPs/IDs for modeling
df = basic_cleaning(df, drop_cols=drop_cols)
df = df.replace([np.inf, -np.inf], np.nan)
df = simple_impute_numeric(df, strategy='median')
print("After cleaning shape:", df.shape)

In [ ]:
# LABELS
if LABEL_COL not in df.columns:
    # try some common alternatives
    possible = [c for c in df.columns if 'label' in c.lower() or 'attack' in c.lower()]
    if possible:
        LABEL_COL = possible[0]
        print("Using label column:", LABEL_COL)
    else:
        raise ValueError("No label column found. Inspect your CSV's column names.")


In [ ]:
# Quick class balance
plot_class_balance(df, label_col=LABEL_COL)

In [ ]:
# SELECT FEATURES & TARGET
# drop non-numeric columns except label
numeric = df.select_dtypes(include=[np.number]).columns.tolist()
if LABEL_COL in numeric:
    numeric.remove(LABEL_COL)
X = df[numeric].copy()
y = df[LABEL_COL].copy()

In [ ]:
# Optionally reduce features: pick top features by variance to keep demo small
top_k = 50 if X.shape[1] > 50 else X.shape[1]
top_feats = X.var().sort_values(ascending=False).head(top_k).index.tolist()
X = X[top_feats]
print("Using features:", len(X.columns))

In [ ]:
plot_corr_heatmap(pd.concat([X, y], axis=1), top_k=30)

In [ ]:
# train/test split 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    stratify=y, random_state=RANDOM_STATE)


In [ ]:
#  scale features 
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
# Save scaler for reproducibility
import joblib
joblib.dump(scaler, "scaler.joblib")

In [ ]:
# Baseline ML: RandomForest 
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_s, y_train)
y_pred = rf.predict(X_test_s)

In [ ]:
print("RandomForest accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))